# Supermarket Sales Analysis
**AI + Data: Make Data Intelligent — Masterclass 1**

This notebook covers dataset understanding, data-quality review, cleaning, business analysis, visualisations and recommendations.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

FILE = "AI + Data: Make Data Intelligent | Masterclass 1 | Practice Dataset.xlsx"
df = pd.read_excel(FILE, sheet_name="Raw_Data")
print("Rows:", len(df))
print("Columns:", list(df.columns))
df.head()

In [ ]:
display(df.info())
display(df.describe(include="all").T)
display(pd.DataFrame({"Missing":df.isna().sum(),"Missing %":(df.isna().mean()*100).round(2)}))
print("Exact duplicate rows:", df.duplicated().sum())
print("Negative quantities:", (df.Quantity<0).sum())
print("Negative profits:", (df.Profit<0).sum())

## Data cleaning

In [ ]:
def parse_date(x):
    if pd.isna(x): return pd.NaT
    if isinstance(x,(pd.Timestamp,np.datetime64)): return pd.to_datetime(x)
    sx=str(x).strip()
    for fmt in ("%Y-%m-%d %H:%M:%S","%Y-%m-%d","%d-%m-%Y"):
        try: return pd.to_datetime(sx,format=fmt)
        except: pass
    return pd.NaT

clean=df.drop_duplicates().copy()
clean["Order_Date_Clean"]=clean["Order_Date"].apply(parse_date)
clean["Product"]=clean["Product"].astype("string").str.strip()
clean["Category"]=clean["Category"].astype("string").str.strip().str.title()
clean["Region"]=clean["Region"].astype("string").str.strip().str.title()
clean["Revenue"]=pd.to_numeric(clean["Revenue"].astype("string").str.replace(r"[^\d.\-]","",regex=True),errors="coerce")

analysis=clean[
    clean["Order_Date_Clean"].notna() & clean["Product"].notna() & clean["Region"].notna() &
    clean["Quantity"].notna() & (clean["Quantity"]>0) & clean["Revenue"].notna() & clean["Profit"].notna()
].copy()
analysis["Month"]=analysis["Order_Date_Clean"].dt.to_period("M").astype(str)
analysis["Year"]=analysis["Order_Date_Clean"].dt.year
counts=analysis["Customer_ID"].value_counts(dropna=True)
analysis["Customer_Segment"]=analysis["Customer_ID"].map(counts).map(lambda x:"Repeat Customer" if x>1 else ("New Customer" if pd.notna(x) else "Unknown"))
analysis["Order_Value_Category"]=pd.qcut(analysis["Revenue"],q=3,labels=["Low","Medium","High"],duplicates="drop")
print("Core analysis rows:",len(analysis))

## Business analysis

In [ ]:
product=analysis.groupby("Product").agg(Revenue=("Revenue","sum"),Profit=("Profit","sum"),Quantity=("Quantity","sum")).sort_values("Revenue",ascending=False)
category=analysis.groupby("Category").agg(Revenue=("Revenue","sum"),Profit=("Profit","sum"),Quantity=("Quantity","sum")).sort_values("Revenue",ascending=False)
region=analysis.groupby("Region").agg(Revenue=("Revenue","sum"),Profit=("Profit","sum"),Quantity=("Quantity","sum")).sort_values("Revenue",ascending=False)
segment=analysis.groupby("Customer_Segment").agg(Revenue=("Revenue","sum"),Orders=("Order_ID","count"),Average_Order_Value=("Revenue","mean")).sort_values("Revenue",ascending=False)
monthly=analysis.groupby("Month").agg(Revenue=("Revenue","sum"),Profit=("Profit","sum")).reset_index()

print("Top products"); display(product.head(10))
print("Category performance"); display(category)
print("Region performance"); display(region)
print("Customer segments"); display(segment)
print("Monthly performance"); display(monthly)

In [ ]:
fig,ax=plt.subplots(figsize=(8,4.5)); category.Revenue.sort_values().plot(kind="barh",ax=ax); ax.set_title("Revenue by Category"); ax.set_xlabel("Revenue"); plt.tight_layout(); plt.show()
fig,ax=plt.subplots(figsize=(8,4.5)); region.Revenue.sort_values().plot(kind="barh",ax=ax); ax.set_title("Revenue by Region"); ax.set_xlabel("Revenue"); plt.tight_layout(); plt.show()
fig,ax=plt.subplots(figsize=(8,4.5)); product.head(10).Revenue.sort_values().plot(kind="barh",ax=ax); ax.set_title("Top 10 Products by Revenue"); ax.set_xlabel("Revenue"); plt.tight_layout(); plt.show()
fig,ax=plt.subplots(figsize=(8,4.5)); ax.plot(monthly.Month,monthly.Revenue,marker="o"); ax.set_title("Monthly Revenue"); ax.set_xlabel("Month"); ax.set_ylabel("Revenue"); plt.xticks(rotation=45); plt.tight_layout(); plt.show()

## 12 management questions

**Sales:** highest-revenue month; highest-profit month; average revenue per valid order.

**Product:** highest-revenue product; highest-profit product; highest-revenue category.

**Customer:** repeat vs new customer revenue; highest average order value segment; repeat-order share.

**Region:** highest-revenue region; highest-profit region; highest average revenue per valid order.

These questions use only fields available in the dataset.

In [ ]:
print("Valid orders:",len(analysis))
print("Total revenue:",analysis.Revenue.sum())
print("Total profit:",analysis.Profit.sum())
print("Top product:",product.index[0])
print("Top category:",category.index[0])
print("Top region:",region.index[0])
print("Repeat customer revenue:",segment.loc["Repeat Customer","Revenue"])

## Recommendations
- Prioritise inventory and promotions around strong products.
- Protect Electronics availability and monitor its profit margin.
- Study Pune's successful product/customer mix and apply learnings to weaker regions.
- Focus on retention and repeat-purchase campaigns because repeat customers drive most revenue.
- Review negative quantities and negative-profit transactions as possible returns, corrections or loss-making orders.
- Add transaction type/return status, payment method and customer rating in future data collection if those metrics are required.